<a href="https://colab.research.google.com/github/BarshaPanthi7/Danphe_task_Barsha/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

In [2]:
import os, shutil
os.makedirs("/root/.config/kaggle", exist_ok=True)
shutil.move("/content/kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)
print("Kaggle API configured successfully.")

Kaggle API configured successfully.


In [3]:
!pip install -q --upgrade kaggle
!kaggle --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 12.4 MB/s eta 0:00:00
Kaggle CLI 2.2.4


ptb xl dataset metadata

In [4]:
!kaggle datasets files -d physionet/ptbxl-electrocardiography-database --page-size 200

Next Page Token = CfDJ8GNxPX3qpaJEpxfoBSRBZPek8iEc_Ol1whDD8EgBVWhoS1biCpsN5yCgAbzXHYEznacAn1jE1j9y0JLCdYYBK0Dv4E63hEIGrDZHtsyl6vIJPZyaY8ues5Meed11l69FZj-U27pjtg
name                size  creationDate                
----------------  ------  --------------------------  
WFDB/HR00001.hea     666  2020-06-15 08:12:13.856000  
WFDB/HR00001.mat  120024  2020-06-15 08:12:13.794000  
WFDB/HR00002.hea     664  2020-06-15 08:12:13.787000  
WFDB/HR00002.mat  120024  2020-06-15 08:12:13.817000  
WFDB/HR00003.hea     661  2020-06-15 08:12:13.827000  
WFDB/HR00003.mat  120024  2020-06-15 08:12:13.785000  
WFDB/HR00004.hea     663  2020-06-15 08:12:13.759000  
WFDB/HR00004.mat  120024  2020-06-15 08:12:13.762000  
WFDB/HR00005.hea     658  2020-06-15 08:12:13.806000  
WFDB/HR00005.mat  120024  2020-06-15 08:12:13.827000  
WFDB/HR00006.hea     659  2020-06-15 08:12:13.771000  
WFDB/HR00006.mat  120024  2020-06-15 08:12:13.837000  
WFDB/HR00007.hea     660  2020-06-15 08:12:13.782000  
WFDB/HR00007.m

downloading metadata files only

In [5]:
import kagglehub
path = kagglehub.dataset_download('physionet/ptbxl-electrocardiography-database')

100%|██████████| 1.35G/1.35G [00:16<00:00, 87.9MB/s]

Extracting files...


inspecting structure

In [12]:
print("Contents of path:", os.listdir(path))

Contents of path: ['WFDB']


In [13]:
import os

wfdb_path = os.path.join(path, 'WFDB')
files = os.listdir(wfdb_path)
hea_files = [f for f in files if f.endswith('.hea')]
print("Number of header files:", len(hea_files))
print("Example files:", hea_files[:5])

Number of header files: 21837
Example files: ['HR06419.hea', 'HR12367.hea', 'HR01208.hea', 'HR12592.hea', 'HR16882.hea']


So WFDB is a subfolder inside the PTB-XL dataset folder (using the standard WFDB format, which stands for WaveForm DataBase - a common format for storing physiological signal data like ECGs).

Each record has two files:

.hea (header) - the metadata: patient age/sex, diagnosis code, sampling rate, lead names, etc. (plain text, human-readable - this is what you printed earlier)

.mat (MATLAB format) - the actual raw ECG waveform signal data (binary, not human-readable directly - this is the heavy data, 120KB+ per file)

In [14]:
with open(os.path.join(wfdb_path, hea_files[0])) as f:
    print(f.read())

HR06419 12 500 5000 04-Jun-2020 15:11:55
HR06419.mat 16+24 200/mV 16 0 95 10649 0 I
HR06419.mat 16+24 200/mV 16 0 90 6455 0 II
HR06419.mat 16+24 200/mV 16 0 -5 -4126 0 III
HR06419.mat 16+24 200/mV 16 0 -92 -8524 0 aVR
HR06419.mat 16+24 200/mV 16 0 50 7755 0 aVL
HR06419.mat 16+24 200/mV 16 0 42 1408 0 aVF
HR06419.mat 16+24 200/mV 16 0 60 154 0 V1
HR06419.mat 16+24 200/mV 16 0 35 -17718 0 V2
HR06419.mat 16+24 200/mV 16 0 50 -17751 0 V3
HR06419.mat 16+24 200/mV 16 0 -50 -11997 0 V4
HR06419.mat 16+24 200/mV 16 0 -475 -11162 0 V5
HR06419.mat 16+24 200/mV 16 0 2925 -31664 0 V6
#Age: 81
#Sex: Female
#Dx: 426783006
#Rx: Unknown
#Hx: Unknown
#Sx: Unknown



###  Parsing metadata from header files

Each `.hea` file contains per-record metadata (age, sex, diagnosis code, sampling rate) embedded as plain text. Since this dataset does not include a separate metadata CSV, metadata is extracted directly from the header files.

In [19]:
import re
import pandas as pd

records = []

for fname in hea_files:
    filepath = os.path.join(wfdb_path, fname)
    with open(filepath) as f:
        content = f.read()

    record_id = fname.replace('.hea', '')

    age_match = re.search(r'#Age:\s*(\S+)', content)
    sex_match = re.search(r'#Sex:\s*(\S+)', content)
    dx_match = re.search(r'#Dx:\s*(\S+)', content)

    first_line = content.splitlines()[0].split()
    fs = int(first_line[2])
    n_samples = int(first_line[3])

    records.append({
        'record_id': record_id,
        'age': age_match.group(1) if age_match else None,
        'sex': sex_match.group(1) if sex_match else None,
        'dx_code': dx_match.group(1) if dx_match else None,
        'sampling_rate': fs,
        'n_samples': n_samples
    })

In [20]:
ptbxl_df = pd.DataFrame(records)

In [21]:
print("Shape:", ptbxl_df.shape)

Shape: (21837, 6)


In [22]:
ptbxl_df.head()

,record_id,age,sex,dx_code,sampling_rate,n_samples
0,HR06419,81,Female,426783006,500,5000
1,HR12367,75,Female,"164865005,164951009,426783006,428750005,47665007",500,5000
2,HR01208,43,Male,426783006,500,5000
3,HR12592,65,Male,426783006,500,5000
4,HR16882,70,Male,"164865005,164951009,426783006",500,5000


last 10 rows

In [23]:
ptbxl_df.tail(10)

,record_id,age,sex,dx_code,sampling_rate,n_samples
21827,HR21005,48,Female,"164865005,164951009,426783006",500,5000
21828,HR01514,37,Female,"164934002,426783006,59931005",500,5000
21829,HR00504,29,Female,426783006,500,5000
21830,HR12604,71,Female,426783006,500,5000
21831,HR14664,77,Female,426783006,500,5000
21832,HR19790,61,Male,"164934002,164951009,426783006,713426002",500,5000
21833,HR14825,79,Female,"164934002,427084000,698252002",500,5000
21834,HR02228,86,Male,"164884008,164889003,164934002,428750005,429622005",500,5000
21835,HR19816,55,Male,"164909002,39732003,426783006,67741000119109",500,5000
21836,HR18465,60,Male,10370003,500,5000


missing value check

In [24]:
ptbxl_df.isnull().sum()

,0
record_id,0
age,0
sex,0
dx_code,0
sampling_rate,0
n_samples,0


age distribution

this is a dataset skewed toward older/middle-aged adults (median 62), which makes sense for a cardiology dataset. ECG abnormalities become more common with age. The one outlier (age 2) is worth a quick sanity check you could look at that specific record to see if it's a genuine pediatric case or a data entry anomaly.

In [25]:
ptbxl_df['age'] = pd.to_numeric(ptbxl_df['age'], errors='coerce')
ptbxl_df['age'].describe()

,age
count,21748.000000
mean,59.836307
std,16.953125
min,2.000000
25%,50.000000
50%,62.000000
75%,72.000000
max,95.000000


investigating missing age values

In [28]:
invalid_ages = ptbxl_df[ptbxl_df['age'].isnull()]
print("Number of records with invalid/missing age:", len(invalid_ages))
invalid_ages.head(10)

Number of records with invalid/missing age: 89


,record_id,age,sex,dx_code,sampling_rate,n_samples
232,HR10054,NaN,Female,"164861001,164889003,428750005,429622005,59931005",500,5000
373,HR18156,NaN,Male,"164865005,164947007,426783006",500,5000
388,HR00727,NaN,Female,"164861001,164865005,164873001,164884008,397320...",500,5000
509,HR00646,NaN,Female,"164865005,164934002,284470004,39732003,4267830...",500,5000
652,HR02990,NaN,Female,"164865005,164889003",500,5000
1012,HR07991,NaN,Female,"164865005,164917005,164947007,270492004,426783...",500,5000
1180,HR05694,NaN,Female,"164865005,445118002",500,5000
1231,HR06017,NaN,Female,"164873001,164934002,426783006,429622005",500,5000
1829,HR06548,NaN,Female,"426783006,713426002,89792004",500,5000
2619,HR00712,NaN,Female,"164884008,164909002,39732003,426783006",500,5000


In [29]:
for record_id in invalid_ages['record_id'].head(5):
    filepath = os.path.join(wfdb_path, record_id + '.hea')
    with open(filepath) as f:
        content = f.read()
    age_line = [line for line in content.splitlines() if '#Age' in line]
    print(record_id, '->', age_line)

HR10054 -> ['#Age: NaN']
HR18156 -> ['#Age: NaN']
HR00727 -> ['#Age: NaN']
HR00646 -> ['#Age: NaN']
HR02990 -> ['#Age: NaN']


###  Finding

89 of 21,837 records (0.4%) have age explicitly recorded as "NaN" in the original header files, indicating age was not captured for these recordings rather than a data parsing error. These records are excluded from age based statistics but retain valid sex, diagnosis, and signal data.

The text string "NaN" just three regular characters (N, a, N) sitting in a text field, inside your header files someone/some system wrote the literal word "NaN" as a placeholder when age wasn't recorded, but it was stored as plain text, not as pandas' special missing-value marker.

That's exactly why our first isnull() check showed 0 missing because the column held the text "NaN", not the real missing-value type. Only after converting the column to numbers (pd.to_numeric) did pandas correctly turn that text into genuine NaN, which isnull() could then detect.

Sex distribution

In [30]:
ptbxl_df['sex'].value_counts()

,count
sex,
Male,11379
Female,10458


### Finding

The dataset is roughly balanced by sex: 11,379 male records (52.1%) and 10,458 female records (47.9%). No significant class imbalance is present.